In [1]:
from azure.storage.blob import BlobServiceClient
from io import BytesIO
import pandas as pd
import os
from dotenv import load_dotenv

In [2]:
def blob_to_df(container_client, blob_name):
    client = container_client.get_blob_client(blob_name)
    data = client.download_blob().readall()
    stream = BytesIO(data)
    df = pd.read_csv(stream)
    return df

In [3]:
load_dotenv()

True

In [18]:
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
container_name = "recsys"

blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service_client.get_container_client(container_name)

In [ ]:
user_map_df = pd.DataFrame({
    "name": ['Qiqi','Torid', 'Uche'],
    'user_id': ['U19519', 'U12188', 'U22123']
})
user_map_df

In [ ]:
user_map = "user_map/user_map.csv"

container_client.get_blob_client(user_map).upload_blob(
        user_map_df.to_csv(index=False), overwrite=True)

In [44]:
news = pd.read_csv('news.tsv', header=None, sep='\t') #news.tsv is deleted and just uploaded from local
news.columns = [
    'content_id',
    "category",
    "sub_category",
    "title",
    "abstract",
    "URL",
    "title_entities",
    "abstract_entities"]

news = news.drop(columns=['URL', 'title_entities', 'abstract_entities'])

In [45]:
import re
import wordninja

def format_category(s):
    s = wordninja.split(s)
    s = " ".join(s)
    s = s.title()
    
    if s == 'Foodanddrink':
        s = 'Food and Drink'
    if s == 'Tv':
        s = 'TV'
    return s

def format_subcategory(s):
    s = wordninja.split(s)
    s = " ".join(s)
    s = s.title()

    s = s.replace('Tv', 'TV')
    s = s.replace('Ce Leb', 'Celeb')
    s = s.replace('Diy', 'DIY')
    s = s.replace('Us', 'US')
    s = s.replace('Nfl', 'NFL')
    s = s.replace('Mma', 'MMA')

    if s == 'Icehockey Nhl':
        s = 'Ice Hockey NHL'

    return s

In [46]:
news['category'] = news['category'].apply(format_category)
news['sub_category'] = news['sub_category'].apply(format_subcategory)

In [48]:
news = news.dropna()
news.isnull().sum().any()

False

In [49]:
raw_content = 'content/content.csv'
container_client.get_blob_client(raw_content).upload_blob(
        news.to_csv(index=False), overwrite=True)

{'etag': '"0x8DD92E0809EFC59"',
 'last_modified': datetime.datetime(2025, 5, 14, 12, 11, 48, tzinfo=datetime.timezone.utc),
 'content_md5': bytearray(b'\x0c\xdfAh\xeb\xaf\x98\x7f\x8a\xe0\xa3\xce\x94k5m'),
 'client_request_id': '9c375af2-30bc-11f0-8759-7c1e522d67b3',
 'request_id': '29e45481-301e-0003-29c9-c4614a000000',
 'version': '2025-05-05',
 'version_id': None,
 'date': datetime.datetime(2025, 5, 14, 12, 11, 47, tzinfo=datetime.timezone.utc),
 'request_server_encrypted': True,
 'encryption_key_sha256': None,
 'encryption_scope': None,
 'structured_body': None}

In [35]:
raw_content = 'content/content.csv'
raw_content_df = blob_to_df(container_client, raw_content)
raw_content_df.isnull().sum().any()

False

In [34]:
all_recommendations = 'all_recommendations/all_recommendations.csv'
all_recommendations_df = blob_to_df(container_client, all_recommendations)
all_recommendations_df.isnull().sum().any()

False

In [36]:
merged_recommendations = 'merged_recommendations/merged_recommendations.csv'
merged_recommendations_df = blob_to_df(container_client, merged_recommendations)
merged_recommendations_df.isnull().sum().any()

False